# 🌍 Espiritualidad, Religión & Riqueza Nacional
## *¿El dinero aleja a las personas de Dios?*

---

> **Reto Makeover Monday — Edición 2026**  
> **Storytelling con «Empatía Predictiva»**: El tono narrativo está calibrado hacia la *sorpresa*,  
> resaltando cómo **EE. UU.** rompe la tendencia global que predice la teoría de la secularización.

---

### 📐 Fuentes de datos
| Dataset | Fuente | Año |
|---|---|---|
| Espiritualidad & Religión — % de adultos que… | Pew Research Center | 2023 |
| GDP per cápita, PPP (USD corrientes) | Banco Mundial (WDI) | 2023 |

## 1. Importación de librerías

In [5]:
# pip install plotly

In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print('Librerías cargadas correctamente')

Librerías cargadas correctamente


## 2. Carga y exploración de datos

In [4]:
# ── Espiritualidad & Religión ──────────────────────────────────────────────
df_rel = pd.read_csv('Spirituality & Religion - % of adults who.csv')

# Renombrar columnas para mayor claridad
df_rel.columns = [
    'Country',
    'pct_religiously_affiliated',
    'pct_believe_life_after_death',
    'pct_nature_spirits',
    'pct_pray_daily',
    'pct_fortune_teller'
]

print(f'Shape: {df_rel.shape}')
df_rel.head(10)

Shape: (36, 6)


,Country,pct_religiously_affiliated,pct_believe_life_after_death,pct_nature_spirits,pct_pray_daily,pct_fortune_teller
0,Argentina,76,65.0,65.0,39.0,14.0
1,Australia,48,49.0,48.0,16.0,11.0
2,Bangladesh,100,77.0,38.0,75.0,11.0
3,Brazil,84,66.0,58.0,76.0,10.0
4,Canada,57,56.0,47.0,30.0,10.0
5,Chile,66,67.0,74.0,41.0,14.0
6,Colombia,77,67.0,70.0,71.0,11.0
7,France,53,55.0,52.0,18.0,13.0
8,Germany,57,50.0,49.0,16.0,7.0
9,Ghana,98,74.0,87.0,73.0,16.0


In [6]:
# ── GDP per cápita, PPP — Banco Mundial ────────────────────────────────────
df_gdp_raw = pd.read_csv(
    'API_NY.GDP.PCAP.PP.CD_DS2_en_csv_v2_121708.csv',
    skiprows=4   # las 4 primeras filas son metadatos del Banco Mundial
)

# Nos quedamos solo con lo que necesitamos: código ISO + valor 2023
df_gdp = df_gdp_raw[['Country Name', 'Country Code', '2023']].copy()
df_gdp.rename(columns={'2023': 'gdp_ppp_2023'}, inplace=True)

print(f'Shape GDP: {df_gdp.shape}')
df_gdp.dropna(subset=['gdp_ppp_2023']).head(5)

Shape GDP: (266, 3)


,Country Name,Country Code,gdp_ppp_2023
0,Aruba,ABW,46574.357420
1,Africa Eastern and Southern,AFE,4501.601101
2,Afghanistan,AFG,2201.722907
3,Africa Western and Central,AFW,6520.280313
4,Angola,AGO,9753.600468


## 3. Mapeo de códigos ISO

El dataset de religión usa nombres de país en texto libre. Para hacer el join con el dataset del Banco Mundial necesitamos el código ISO de 3 letras.

In [7]:
ISO3_MAP = {
    'Argentina':    'ARG',
    'Australia':    'AUS',
    'Bangladesh':   'BGD',
    'Brazil':       'BRA',
    'Canada':       'CAN',
    'Chile':        'CHL',
    'Colombia':     'COL',
    'France':       'FRA',
    'Germany':      'DEU',
    'Ghana':        'GHA',
    'Greece':       'GRC',
    'Hungary':      'HUN',
    'India':        'IND',
    'Indonesia':    'IDN',
    'Israel':       'ISR',
    'Italy':        'ITA',
    'Japan':        'JPN',
    'Kenya':        'KEN',
    'Malaysia':     'MYS',
    'Mexico':       'MEX',
    'Netherlands':  'NLD',
    'Nigeria':      'NGA',
    'Peru':         'PER',
    'Philippines':  'PHL',
    'Poland':       'POL',
    'Singapore':    'SGP',
    'South Africa': 'ZAF',
    'South Korea':  'KOR',
    'Spain':        'ESP',
    'Sri Lanka':    'LKA',
    'Sweden':       'SWE',
    'Thailand':     'THA',
    'Tunisia':      'TUN',
    'Turkey':       'TUR',
    'U.S.':         'USA',
    'UK':           'GBR',
}

df_rel['ISO3'] = df_rel['Country'].map(ISO3_MAP)

# Verificar que no haya países sin mapear
missing = df_rel[df_rel['ISO3'].isna()]['Country'].tolist()
if missing:
    print(f'Países sin código ISO: {missing}')
else:
    print(f' Los {len(df_rel)} países tienen código ISO3 asignado')

df_rel[['Country', 'ISO3']].tail(10)

 Los 36 países tienen código ISO3 asignado


,Country,ISO3
26,South Africa,ZAF
27,South Korea,KOR
28,Spain,ESP
29,Sri Lanka,LKA
30,Sweden,SWE
31,Thailand,THA
32,Tunisia,TUN
33,Turkey,TUR
34,U.S.,USA
35,UK,GBR


## 4. Integración de datasets — JOIN por ISO

In [8]:
df = df_rel.merge(
    df_gdp[['Country Code', 'gdp_ppp_2023']],
    left_on='ISO3',
    right_on='Country Code',
    how='left'
).drop(columns='Country Code')

# Eliminar filas sin dato de oración o de GDP
df_clean = df.dropna(subset=['pct_pray_daily', 'gdp_ppp_2023']).copy()

print(f'Filas con datos completos: {len(df_clean)} / {len(df)}')
df_clean[['Country','ISO3','pct_pray_daily','pct_religiously_affiliated','gdp_ppp_2023']]

Filas con datos completos: 35 / 36


,Country,ISO3,pct_pray_daily,pct_religiously_affiliated,gdp_ppp_2023
0,Argentina,ARG,39.0,76,30221.497689
1,Australia,AUS,16.0,48,72272.709214
2,Bangladesh,BGD,75.0,100,9147.777507
3,Brazil,BRA,76.0,84,21175.618699
4,Canada,CAN,30.0,57,64219.317393
5,Chile,CHL,41.0,66,33145.403552
6,Colombia,COL,71.0,77,21246.376270
7,France,FRA,18.0,53,60838.941672
8,Germany,DEU,16.0,57,71684.031188
9,Ghana,GHA,73.0,98,7556.416055


## 5. Ingeniería de variables auxiliares

In [9]:
# ── Etiqueta de región ─────────────────────────────────────────────────────
REGION_MAP = {
    'ARG': 'Latam', 'BRA': 'Latam', 'CHL': 'Latam', 'COL': 'Latam',
    'MEX': 'Latam', 'PER': 'Latam',
    'AUS': 'Anglosfera', 'CAN': 'Anglosfera', 'GBR': 'Anglosfera',
    'USA': 'Anglosfera',
    'FRA': 'Europa', 'DEU': 'Europa', 'GRC': 'Europa', 'HUN': 'Europa',
    'ITA': 'Europa', 'NLD': 'Europa', 'POL': 'Europa', 'ESP': 'Europa',
    'SWE': 'Europa',
    'BGD': 'Asia', 'IND': 'Asia', 'IDN': 'Asia', 'JPN': 'Asia',
    'KOR': 'Asia', 'LKA': 'Asia', 'MYS': 'Asia', 'PHL': 'Asia',
    'SGP': 'Asia', 'THA': 'Asia', 'TUR': 'Asia',
    'GHA': 'Africa', 'KEN': 'Africa', 'NGA': 'Africa', 'ZAF': 'Africa',
    'TUN': 'Africa',
    'ISR': 'Medio Oriente',
}

df_clean['region'] = df_clean['ISO3'].map(REGION_MAP)

# ── Flag para EE.UU. ──────────────────────────────────────────────────────
df_clean['is_us'] = df_clean['ISO3'] == 'USA'

# ── GDP en miles (más legible en ejes) ────────────────────────────────────
df_clean['gdp_k'] = df_clean['gdp_ppp_2023'] / 1_000

print('Variables calculadas correctamente')
df_clean[['Country','region','is_us','gdp_k','pct_pray_daily']].sample(5)

Variables calculadas correctamente


,Country,region,is_us,gdp_k,pct_pray_daily
23,Philippines,Asia,False,10.985829,79.0
15,Italy,Europa,False,60.029872,35.0
21,Nigeria,Africa,False,8.705434,84.0
10,Greece,Europa,False,42.710506,37.0
34,U.S.,Anglosfera,True,82.304620,44.0


## 6. Línea de tendencia (regresión log-lineal)

La teoría de la secularización predice que a mayor riqueza, menor práctica religiosa.  
Ajustamos un modelo `y ~ log(x)` para mostrar la tendencia global.

In [10]:
# Excluir EE.UU. del ajuste para que la línea refleje la tendencia 'normal'
df_fit = df_clean[~df_clean['is_us']]

log_x = np.log(df_fit['gdp_ppp_2023'])
y     = df_fit['pct_pray_daily']

# Regresión de grado 1 en espacio log
coeffs = np.polyfit(log_x, y, 1)
poly   = np.poly1d(coeffs)

# Curva suavizada para dibujar
x_range = np.linspace(df_clean['gdp_ppp_2023'].min(), df_clean['gdp_ppp_2023'].max(), 300)
y_trend = poly(np.log(x_range))

# R²
y_pred = poly(log_x)
ss_res = np.sum((y - y_pred)**2)
ss_tot = np.sum((y - y.mean())**2)
r2 = 1 - ss_res / ss_tot

print(f'Modelo: pray_daily = {coeffs[0]:.2f}*ln(GDP) + {coeffs[1]:.2f}')
print(f'R2 (sin EE.UU.) = {r2:.3f}')

# Desviacion de EE.UU.
us_row       = df_clean[df_clean['is_us']].iloc[0]
us_predicted = poly(np.log(us_row['gdp_ppp_2023']))
us_actual    = us_row['pct_pray_daily']
deviation    = us_actual - us_predicted
print(f'\nEE.UU. real: {us_actual}% | Predicho: {us_predicted:.1f}% | Desviacion: +{deviation:.1f}pp')

Modelo: pray_daily = -26.24*ln(GDP) + 317.75
R2 (sin EE.UU.) = 0.653

EE.UU. real: 44.0% | Predicho: 20.8% | Desviacion: +23.2pp


---
## 7. Visualización Principal — *Outlier de los Ricos Creyentes*

Scatter plot inmersivo con:
- **Eje X**: GDP per cápita PPP 2023 (en miles USD)
- **Eje Y**: % de adultos que oran a diario
- **Color**: Región geográfica
- **Tamaño**: % de adultos con afiliación religiosa
- **Línea de tendencia**: Regresión log-lineal (sin EE.UU.) — la «norma» secular
- **Spotlight**: EE.UU. destacado como outlier narrativo

In [ ]:
# --- Paleta por región ---
PALETTE = {
    'Latam':         '#F97316',
    'Anglosfera':    '#6366F1',
    'Europa':        '#3B82F6',
    'Asia':          '#10B981',
    'Africa':        '#F59E0B',
    'Medio Oriente': '#EC4899',
}

BG       = '#0F172A'
GRID_CLR = '#1E293B'
TEXT_CLR = '#E2E8F0'
MUTED    = '#64748B'
US_CLR   = '#FBBF24'

fig = go.Figure()

# --- 1. Banda de confianza ---
fig.add_trace(go.Scatter(
    x=np.concatenate([x_range/1000, x_range[::-1]/1000]),
    y=np.concatenate([y_trend + 8, (y_trend - 8)[::-1]]),
    fill='toself',
    fillcolor='rgba(99,102,241,0.07)',
    line=dict(color='rgba(0,0,0,0)'),
    hoverinfo='skip',
    showlegend=False
))

# --- 2. Línea de tendencia ---
fig.add_trace(go.Scatter(
    x=x_range / 1000,
    y=y_trend,
    mode='lines',
    line=dict(color='rgba(99,102,241,0.6)', width=2, dash='dot'),
    name=f'Tendencia secular (R²={r2:.2f})',
    hoverinfo='skip',
))

# --- 3. Scatter por región ---
for region, color in PALETTE.items():
    subset = df_clean[(df_clean['region'] == region) & (~df_clean['is_us'])]
    if subset.empty:
        continue

    fig.add_trace(go.Scatter(
        x=subset['gdp_k'],
        y=subset['pct_pray_daily'],
        mode='markers+text',
        marker=dict(
            size=subset['pct_religiously_affiliated'] / 4 + 6,
            color=color,
            opacity=0.82,
            line=dict(color='rgba(255,255,255,0.25)', width=1),
        ),
        text=subset['Country'],
        textposition='top center',
        textfont=dict(size=9, color=color),
        name=region,
        customdata=subset[['pct_religiously_affiliated', 'gdp_ppp_2023', 'region']].values,
        hovertemplate=(
            '<b>%{text}</b><br>'
            'Oración diaria: <b>%{y}%</b><br>'
            'GDP PPP 2023: <b>$%{customdata[1]:,.0f}</b><br>'
            'Afiliación religiosa: <b>%{customdata[0]}%</b><br>'
            'Región: %{customdata[2]}'
            '<extra></extra>'
        ),
    ))

# --- 4. CONFIGURACIÓN DE OUTLIERS ---
OUTLIERS = [
    {
        'iso': 'USA',
        'label': 'EE.UU.',
        'color': US_CLR,
        'symbol': 'star',
        'ax_offset': (-18, 16),
        'label_offset': (-18, 20),
        'narrative': lambda row, pred, dev: (
            f'<b>EE.UU.</b><br>'
            f'{row["pct_pray_daily"]:.0f}% ora diario — +{dev:.0f}pp<br>'
            '<i>sobre la tendencia secular</i>'
        )
    },
    {
        'iso': 'SGP',
        'label': 'Singapur',
        'color': '#34D399',
        'symbol': 'star',
        'ax_offset': (-10, 10),
        'label_offset': (-10, 15),
        'narrative': lambda row, pred, dev: (
            f'<b>Singapur</b><br>'
            f'{row["pct_pray_daily"]:.0f}% ora diario<br>'
            '<i>caso atípico en Asia </i>'
        )
    }
]

# --- 5. FUNCIÓN PARA OUTLIERS ---
def add_outlier(fig, row, config, y_pred=None, deviation=None):
    color = config['color']

    # Flecha
    fig.add_annotation(
        x=row['gdp_k'], y=row['pct_pray_daily'],
        ax=row['gdp_k'] + config['ax_offset'][0],
        ay=row['pct_pray_daily'] + config['ax_offset'][1],
        xref='x', yref='y', axref='x', ayref='y',
        text='',
        showarrow=True,
        arrowhead=2,
        arrowsize=1.2,
        arrowcolor=color,
        arrowwidth=2.2
    )

    # Punto
    fig.add_trace(go.Scatter(
        x=[row['gdp_k']],
        y=[row['pct_pray_daily']],
        mode='markers',
        marker=dict(
            size=row['pct_religiously_affiliated'] / 4 + 10,
            color=color,
            symbol=config['symbol'],
            line=dict(color='white', width=2),
        ),
        name=f"{config['label']} (outlier)",
    ))

    # Label
    text = config['narrative'](row, y_pred, deviation)

    fig.add_annotation(
        x=row['gdp_k'] + config['label_offset'][0],
        y=row['pct_pray_daily'] + config['label_offset'][1],
        text=text,
        font=dict(size=10, color=color),
        bgcolor='rgba(255,255,255,0.05)',
        bordercolor=color,
        borderwidth=1.5,
        borderpad=6,
        showarrow=False,
        align='left'
    )

# --- 6. APLICAR OUTLIERS ---
for cfg in OUTLIERS:
    row = df_clean[df_clean['ISO3'] == cfg['iso']].iloc[0]

    if cfg['iso'] == 'USA':
        y_pred = us_predicted
        dev = deviation
    else:
        y_pred = None
        dev = None

    add_outlier(fig, row, cfg, y_pred, dev)

fig.update_layout(
    template='plotly_dark',
    paper_bgcolor=BG,
    plot_bgcolor=BG,
    width=1050, height=680,

    title=dict(
        text=(
            '<b> Outlier de los Ricos Creyentes</b><br>'
            '<span style="font-size:13px;color:#94A3B8;">'
            'GDP per cápita PPP vs. oración diaria — 35 países · 2023</span>'
        ),
        x=0.5,
        font=dict(size=22, color=TEXT_CLR),
    ),

    xaxis=dict(
        title=dict(
            text='<b>GDP per cápita PPP 2023</b> (miles de USD)',
            font=dict(size=13, color=MUTED)
        ),
        tickprefix='$',
        ticksuffix='K',
        gridcolor=GRID_CLR,
        zeroline=False,
        tickfont=dict(color=MUTED),
        showspikes=True,
        spikecolor=MUTED,
        spikethickness=1
    ),

    yaxis=dict(
        title=dict(
            text='<b>% de adultos que oran a diario</b>',
            font=dict(size=13, color=MUTED)
        ),
        ticksuffix='%',
        gridcolor=GRID_CLR,
        zeroline=False,
        tickfont=dict(color=MUTED),
        showspikes=True,
        spikecolor=MUTED,
        spikethickness=1
    ),

    legend=dict(
        x=1, y=1,
        xanchor='right',
        bgcolor='rgba(15,23,42,0.7)',
        bordercolor=GRID_CLR,
        borderwidth=1,
        font=dict(size=11, color=TEXT_CLR),
    ),

    margin=dict(l=70, r=40, t=100, b=70),
)

---
## 8. Visualización Complementaria — *Ranking comparativo*

Barras horizontales duales: oración diaria vs afiliación religiosa por país.

In [22]:
df_sorted = df_clean.sort_values('pct_pray_daily', ascending=True).copy()
colors_bar = [US_CLR if iso == 'USA' else '#3B82F6' for iso in df_sorted['ISO3']]

fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=('<b>% oran a diario</b>', '<b>% afiliación religiosa</b>'),
    horizontal_spacing=0.04
)

fig2.add_trace(go.Bar(
    y=df_sorted['Country'],
    x=df_sorted['pct_pray_daily'],
    orientation='h',
    marker_color=colors_bar,
    marker_line_width=0,
    text=df_sorted['pct_pray_daily'].map(lambda v: f'{v:.0f}%'),
    textposition='outside',
    textfont=dict(size=9, color=TEXT_CLR),
    hovertemplate='<b>%{y}</b><br>Oración diaria: %{x}%<extra></extra>',
    name='', showlegend=False,
), row=1, col=1)

fig2.add_trace(go.Bar(
    y=df_sorted['Country'],
    x=df_sorted['pct_religiously_affiliated'],
    orientation='h',
    marker_color=colors_bar,
    marker_opacity=0.6,
    marker_line_width=0,
    text=df_sorted['pct_religiously_affiliated'].map(lambda v: f'{v:.0f}%'),
    textposition='outside',
    textfont=dict(size=9, color=TEXT_CLR),
    hovertemplate='<b>%{y}</b><br>Afiliación: %{x}%<extra></extra>',
    name='', showlegend=False,
), row=1, col=2)

fig2.update_layout(
    template='plotly_dark',
    paper_bgcolor=BG,
    plot_bgcolor=BG,
    height=820, width=1050,
    title=dict(
        text='<b>Religiosidad comparada</b> — Oración diaria vs Afiliación religiosa<br>'
             '<span style="font-size:12px;color:#94A3B8;">EE.UU. (dorado) — el outlier que no sigue la norma secular</span>',
        x=0.5, xanchor='center',
        font=dict(size=18, color=TEXT_CLR)
    ),
    margin=dict(l=110, r=50, t=90, b=50)
)
fig2.update_xaxes(gridcolor=GRID_CLR, ticksuffix='%', tickfont=dict(color=MUTED))
fig2.update_yaxes(gridcolor=GRID_CLR, tickfont=dict(color=TEXT_CLR, size=9))

fig2.show()
print('Ranking renderizado')

Ranking renderizado


---
## 9. Exportar datos procesados

In [25]:
cols_export = [
    'Country', 'ISO3', 'region',
    'pct_pray_daily', 'pct_religiously_affiliated',
    'pct_believe_life_after_death', 'pct_nature_spirits',
    'pct_fortune_teller', 'gdp_ppp_2023'
]

df_export = df_clean[cols_export].round(2)
df_export.to_csv('religion_gdp_merged_2023.csv', index=False)

print('Datos exportados → religion_gdp_merged_2023.csv')
print(f'{len(df_export)} países x {len(cols_export)} variables')
df_export.head()

Datos exportados → religion_gdp_merged_2023.csv
35 países x 9 variables


,Country,ISO3,region,pct_pray_daily,pct_religiously_affiliated,pct_believe_life_after_death,pct_nature_spirits,pct_fortune_teller,gdp_ppp_2023
0,Argentina,ARG,Latam,39.0,76,65.0,65.0,14.0,30221.50
1,Australia,AUS,Anglosfera,16.0,48,49.0,48.0,11.0,72272.71
2,Bangladesh,BGD,Asia,75.0,100,77.0,38.0,11.0,9147.78
3,Brazil,BRA,Latam,76.0,84,66.0,58.0,10.0,21175.62
4,Canada,CAN,Anglosfera,30.0,57,56.0,47.0,10.0,64219.32


---
## 10. Notas de Storytelling

### ¿Por qué EE.UU. es el outlier perfecto?

La **teoría de la secularización** predice que a mayor riqueza, menor práctica religiosa. La línea de tendencia confirma ese patrón con solidez (R² ≈ 0.65 sin EE.UU.).

Sin embargo, **EE.UU. se desvía sistemáticamente** del modelo en ~20 puntos porcentuales:

| Métrica | EE.UU. | Pares ricos (promedio) |
|---|---|---|
| Oración diaria | **44%** | ~18–21% |
| Afiliación religiosa | **69%** | ~47–57% |
| GDP PPP per cápita | $82K | $60K–82K |

### Factores explicativos
1. **Mercado religioso desregulado** — competencia entre denominaciones mantiene alta la demanda
2. **Religión e identidad nacional** — el evangelismo está entrelazado con el patriotismo
3. **Desigualdad interna** — bolsas de pobreza e inseguridad que alimentan la práctica religiosa
4. **Inmigración continua** — flujo de inmigrantes con alta religiosidad

---
*Notebook creado por Juan Sebastián Segura para Alegra*